In [1]:
# @title Install dash (if necessary)
%pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 40.8 MB/s eta 0:00:00


In [2]:
# @title Import libraries
import requests
import pandas as pd
import geopandas as gpd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State

In [3]:
# @title Fetching typhoon data as geojson

#fetching geojson
url = "https://agora.ex.nii.ac.jp/digital-typhoon/geojson/wnp/202518.en.json"
response = requests.get(url)
typhoon_geojson = response.json()
typhoon_data = gpd.GeoDataFrame.from_features(typhoon_geojson['features'])

#setting coordinate reference system
typhoon_data.set_crs(epsg=4326, inplace=True)
typhoon_data['longitude'] = typhoon_data.geometry.x
typhoon_data['latitude'] = typhoon_data.geometry.y

#expanding points to areas by setting a buffer based on the 'class' of typhoon at each point (requires a different crs)
typhoon_data.to_crs(epsg=32634, inplace=True)

typhoon_data['geometry'] = typhoon_data.geometry.buffer(typhoon_data['class'] * 100000)
typhoon_data.to_crs(epsg=4326, inplace=True)

#formatting datetime
typhoon_data['display_time'] = pd.to_datetime(typhoon_data['display_time'])
typhoon_data['display_time'] = typhoon_data['display_time'].dt.tz_localize(None)

#previewing data
typhoon_data.head(10)

,geometry,time,wind,display_time,class,pressure,longitude,latitude
0,"POLYGON ((135.82283 12.92456, 135.78927 12.999...",1758067200,30,2025-09-17 00:00:00,2,1006,136.6,13.3
1,"POLYGON ((135.2396 13.11706, 135.2052 13.19075...",1758078000,30,2025-09-17 03:00:00,2,1006,136.0,13.5
2,"POLYGON ((135.04475 13.31087, 135.00968 13.384...",1758088800,30,2025-09-17 06:00:00,2,1004,135.8,13.7
3,"POLYGON ((134.55905 13.40644, 134.52347 13.478...",1758099600,30,2025-09-17 09:00:00,2,1004,135.3,13.8
4,"POLYGON ((133.7826 13.40382, 133.74667 13.4734...",1758110400,30,2025-09-17 12:00:00,2,1004,134.5,13.8
5,"POLYGON ((133.58783 13.59759, 133.55122 13.666...",1758121200,30,2025-09-17 15:00:00,2,1004,134.3,14.0
6,"POLYGON ((133.29464 14.17988, 133.25614 14.248...",1758132000,30,2025-09-17 18:00:00,2,1004,134.0,14.6
7,"POLYGON ((133.00321 14.27609, 132.96428 14.343...",1758142800,30,2025-09-17 21:00:00,2,1004,133.7,14.7
8,"POLYGON ((132.42005 14.56571, 132.37996 14.631...",1758153600,30,2025-09-18 00:00:00,2,1004,133.1,15.0
9,"POLYGON ((132.22569 14.66224, 132.18521 14.727...",1758164400,30,2025-09-18 03:00:00,2,1004,132.9,15.1


In [4]:
# @title Fetching language use data for Philippines from language use data platform as geojson.

#initializing dataframe and API call
language_data = gpd.GeoDataFrame()
page_num = 0
base_url = "https://ludp.clearglobal.org/public/location/PHL?aggregation=2&output=geojson&fields=proportion_value,individuals_value_weighted,language_rank,language_name,language_code,location_name,location_code,location_level,dataset_name,url,source,datetime_published,reliability_score&page="

#looping through pages of API response and adding to language_data dataframe
while True:
    url = f"{base_url}{page_num}"
    response = requests.get(url)
    language_geojson = response.json()
    if len(language_geojson['features']) == 0:
        print(f"End at {page_num}")
        break
    language_data_page = gpd.GeoDataFrame.from_features(
        language_geojson['features'])
    page_num += 1
    language_data = pd.concat([language_data_page, language_data])

#setting geometry and coordinate reference system
language_data = language_data.set_geometry('geometry')
language_data = language_data.set_crs(epsg=4326)

#previewing data
language_data.head(10)

End at 28


,geometry,proportion_value,individuals_value_weighted,language_rank,language_name,language_code,location_name,location_code,location_level,dataset_name,url,source,datetime_published,reliability_score
0,"MULTIPOLYGON (((124.69875 9.25686, 124.70225 9...",0.000675,57,6,Hiligaynon,hili1240,Camiguin,PH10018,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
1,"MULTIPOLYGON (((124.69875 9.25686, 124.70225 9...",0.001634,137,5,Tagalog,taga1270,Camiguin,PH10018,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
2,"MULTIPOLYGON (((124.69875 9.25686, 124.70225 9...",0.004076,342,3,Cinamiguin Manobo,cina1236,Camiguin,PH10018,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
3,"MULTIPOLYGON (((124.69875 9.25686, 124.70225 9...",0.000368,31,7,Caviteño,cavi1254,Camiguin,PH10018,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
4,"MULTIPOLYGON (((126.36835 7.91881, 126.3655 7....",0.000070,30,19,Tadyawan,tady1237,Surigao del Sur,PH16068,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
5,"MULTIPOLYGON (((126.36835 7.91881, 126.3655 7....",0.000458,199,12,Davawenyo,dava1245,Surigao del Sur,PH16068,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
6,"MULTIPOLYGON (((126.36835 7.91881, 126.3655 7....",0.476943,206886,1,Cebuano,cebu1242,Surigao del Sur,PH16068,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
7,"MULTIPOLYGON (((126.36835 7.91881, 126.3655 7....",0.248826,107934,2,Surigaonon,suri1273,Surigao del Sur,PH16068,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
8,"MULTIPOLYGON (((126.36835 7.91881, 126.3655 7....",0.000217,94,15,Tausug,taus1251,Surigao del Sur,PH16068,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883
9,"MULTIPOLYGON (((126.36835 7.91881, 126.3655 7....",0.000212,92,16,Higaonon,higa1237,Surigao del Sur,PH16068,2,Philippines Census 2010 (IPUMS extract),https://api.ipums.org/downloads/ipumsi/api/v1/...,IPUMS International,12-31-2010,0.883


In [5]:
# @title Merging and preparing data

#merging data with spatial join
merged_data = gpd.sjoin(language_data,typhoon_data)
merged_data['language_name'] = merged_data['language_name'].str[:20]
#filtering typhoon data for relevant points
typhoon_data_filter = typhoon_data[ typhoon_data.display_time > '2025-09-21 00:00:00']
#typhoon_data[typhoon_data.index.isin(merged_data.index_right.unique())]


# Get all unique (original) language names for consistent coloring
all_languages = merged_data['language_name'].unique()

# Get a qualitative color scale
colors = px.colors.qualitative.Plotly
# Create a dictionary mapping each language to a color
language_color_map = {lang: colors[i % len(colors)] for i, lang in enumerate(all_languages)}


In [6]:
typhoon_data_filter

,geometry,time,wind,display_time,class,pressure,longitude,latitude
33,"POLYGON ((125.40157 16.86269, 125.27483 16.976...",1758423600,90,2025-09-21 03:00:00,5,940,126.6,18.2
34,"POLYGON ((125.12293 17.04673, 124.9944 17.1587...",1758434400,100,2025-09-21 06:00:00,5,925,126.3,18.4
35,"POLYGON ((124.75199 17.2301, 124.62158 17.3394...",1758445200,100,2025-09-21 09:00:00,5,925,125.9,18.6
36,"POLYGON ((124.28879 17.41283, 124.15641 17.518...",1758456000,100,2025-09-21 12:00:00,5,925,125.4,18.8
37,"POLYGON ((123.73339 17.59498, 123.59895 17.696...",1758466800,100,2025-09-21 15:00:00,5,925,124.8,19.0
38,"POLYGON ((123.08654 17.68353, 122.9507 17.7806...",1758477600,100,2025-09-21 18:00:00,5,925,124.1,19.1
39,"POLYGON ((122.62438 17.77347, 122.48735 17.867...",1758488400,100,2025-09-21 21:00:00,5,925,123.6,19.2
40,"POLYGON ((122.06937 17.95598, 121.93029 18.045...",1758499200,110,2025-09-22 00:00:00,5,905,123.0,19.4
41,"POLYGON ((121.4235 17.95191, 121.28383 18.0366...",1758510000,110,2025-09-22 03:00:00,5,905,122.3,19.4
42,"POLYGON ((120.86993 17.94858, 120.72977 18.029...",1758520800,110,2025-09-22 06:00:00,5,905,121.7,19.4


In [10]:
# @title Preparing dashboard to visualise data

#initialising dashboard
app = dash.Dash(__name__)

#setting the layout
app.layout = html.Div(
    [
        html.Div("Languages used by people affected by typhoon Ragasa", style={'backgroundColor': 'white','fontFamily': 'sans-serif','font-size':'2em'}),

        html.Div(["30 knot wind circle (approx.). Sources: ",dash.dcc.Link("KITAMOTO Asanobu @ National Institute of Informatics",href="https://agora.ex.nii.ac.jp/digital-typhoon/summary/wnp/s/202518.html.en"),", ",dash.dcc.Link("CLEAR Global Language Use Data Platform",href="https://clearglobal.org/language-use-data-platform/?dash=LocationDashboard&country=Philippines")],style={'backgroundColor': 'white','fontFamily': 'sans-serif','margin-bottom': '10px'}),
        html.Div(""),

        html.Div([
            dcc.Interval(
                id='interval-component',
                interval=2000,
                n_intervals=0,
                disabled=True # Start disabled
            ),
            html.Button('▶', id='play-pause-button', n_clicks=0,style={'width': '40px', 'height': '40px', 'borderRadius': '50%', 'padding': '0', 'textAlign': 'center', 'lineHeight': '40px', 'border': 'none', 'backgroundColor': '#e7e7e7', 'color': 'black'}),

            html.Div([html.Div("Date/Time", style={'margin-bottom': '10px','fontFamily': 'sans-serif','font-size':'0.8em'}),
            dcc.Slider(
                id='time-slider',
                min=0,
                max=len(typhoon_data_filter['display_time'].dt.strftime("%d/%m, %H:%M").unique())-1,
                step=1,
                value=0,  # Initial value
                marks={i: {'label': display_time, 'style': { 'whiteSpace': 'nowrap', 'fontFamily': 'sans-serif'}} for i, display_time in enumerate(
                    typhoon_data_filter['display_time'].dt.strftime("%d/%m, %H:%M").unique())}
            )],style={'backgroundColor': 'white','width': '80vw','margin-left' : '50px','margin-right' : '25px'}),

        ],
            style={'backgroundColor': 'white','width': '95vw','margin-left' : '25px','margin-right' : '25px','display': 'flex'}),
        html.Div(
            [
                dcc.Graph(id='typhoon-graph',
                          style={'width': '50%', 'display': 'inline-block',"border":"10px white solid",'height': '500px'},config={
        'displayModeBar': False
    }),
                dcc.Graph(id='language-graph',
                          style={'width': '50%', 'display': 'inline-block',"border":"10px white solid",'height': '550px'},config={
        'displayModeBar': False
    })
            ],
            style={'display': 'flex'}
        )
    ],style={'backgroundColor':'white'}
)

#setting callbacks to update chart based on slider
@app.callback(
    [Output('typhoon-graph', 'figure'), Output('language-graph', 'figure')],
    [Input('time-slider', 'value'), Input('typhoon-graph', 'clickData')]
)

#function to update chart and map based on slider
def update_graphs(selected_time_index, clickData):

    # get value from slider
    selected_display_time = typhoon_data_filter['display_time'].dt.strftime("%d/%m, %H:%M").unique()[
        selected_time_index]

    # Filter data for the typhoon map
    filtered_typhoon_data = typhoon_data_filter[typhoon_data_filter['display_time'].dt.strftime("%d/%m, %H:%M")
                                                == selected_display_time]

    # Calculate center of the filtered typhoon data
    center_lat = filtered_typhoon_data.latitude.mean()
    center_lon = filtered_typhoon_data.longitude.mean()


    # Update typhoon map figure
    typhoon_fig = px.choropleth_map(
        filtered_typhoon_data,
        geojson=filtered_typhoon_data.geometry,
        locations=filtered_typhoon_data.index,
        color="class",
        color_continuous_scale="temps",
        color_continuous_midpoint=3.5,
        range_color=[2, 5],
        map_style="carto-positron",
        zoom=4,
        center={"lat":center_lat, "lon":center_lon},
        opacity=0.5,
        hover_data=['class', 'display_time']
    )
    typhoon_fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        coloraxis_colorbar_x=-0.15,
        title='Position and class of typhoon Ragasa at ' + selected_display_time,
        coloraxis_colorbar=dict(
            title='Class'
        )
    )

    if clickData is None:
        # Filter by slider value
        filtered_data = merged_data[merged_data['display_time'].dt.strftime("%d/%m, %H:%M")
                                    == selected_display_time]
        title_text = 'Top 10 languages in districts affected at ' + selected_display_time
    else:
        display_time = clickData['points'][0]['customdata'][1]
        filtered_data = merged_data[merged_data['display_time']
                                    == display_time]
        title_text = 'Top 10 languages in districts affected at ' + display_time + \
            '(' + ' ,'.join(filtered_data['location_name'].unique()) + ')'

    filtered_data = filtered_data.groupby(['language_name']).agg(
        {'individuals_value_weighted': 'sum'}).reset_index()
    filtered_data['rank'] = filtered_data['individuals_value_weighted'].rank(
        ascending=False)
    total = filtered_data['individuals_value_weighted'].sum()
    filtered_data['proportion_population']= (filtered_data['individuals_value_weighted'] / total) * 100

    filtered_data = filtered_data.sort_values(['individuals_value_weighted'], ascending=False)
    filtered_data = filtered_data[filtered_data['rank'] <= 10]

    lang_fig_updated = px.bar(filtered_data, y='language_name', x='proportion_population', color='language_name',
                              labels={'language_name': 'Language', 'proportion_population': 'Population %'},color_discrete_map=language_color_map)
    lang_fig_updated.update_layout(title=title_text,xaxis=dict(range=[0, 100]), margin=dict(l=200),showlegend=False)
    return typhoon_fig, lang_fig_updated


# Callback to toggle the animation
@app.callback(
    Output('interval-component', 'disabled'),
    [Input('play-pause-button', 'n_clicks')],
    [State('interval-component', 'disabled')]
)
def toggle_animation(n_clicks, currently_disabled):
    if n_clicks:
        return not currently_disabled
    return currently_disabled

# Callback to update the slider based on the interval
@app.callback(
    Output('time-slider', 'value'),
    [Input('interval-component', 'n_intervals')],
    [State('time-slider', 'max')],
    prevent_initial_call=True # Prevent the callback from firing on page load
)
def update_slider(n_intervals, max_value):
    # Calculate the next value for the slider, cycling back to 0 at the end
    next_value = n_intervals % (max_value + 1)
    return next_value

# New callback to update button text based on animation state
@app.callback(
    Output('play-pause-button', 'children'),
    [Input('interval-component', 'disabled')]
)
def update_button_text(disabled):
    if disabled:
        return '▶'
    else:
        return '❚❚'


if __name__ == '__main__':
    #app.run(debug=True)  #uncomment to view in notebook
      app.run(jupyter_mode="external")

Dash app running on:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>